In [132]:
import pandas as pd
import glob

## Cleaning the DF's
Currently, we have 6 data frames across various years; we want to combine each of the data frames into one large data frame. Before that we have to do conversions on the "Date" columns and also add new
columns to see if Atletico-MG won or lost.



In [133]:
#Read all of the CSV's in data and make combine into one large DF for easier analysis
csv_files = glob.glob("data/*[0-9][0-9]-[0-9][0-9].csv")
dfs = []
for file in csv_files:
    season = file.replace(".csv", "").replace("data\\", "")
    df_temp = pd.read_csv(file)
    df_temp["Season"] = season
    dfs.append(df_temp)    


In [134]:
df = pd.concat(dfs)

In [135]:
df.head()

,Date,Home Team,Away Team,Score,Time,Competition,Season
0,"Mon, Jan 11",Red Bull Bragantino,Atlético-MG,2 - 2,FT,Brazilian Serie A,2020-21
1,"Mon, Nov 2",Palmeiras,Atlético-MG,3 - 0,FT,Brazilian Serie A,2020-21
2,"Mon, Oct 19",Bahia,Atlético-MG,3 - 1,FT,Brazilian Serie A,2020-21
3,"Sat, Aug 22",Internacional,Atlético-MG,1 - 0,FT,Brazilian Serie A,2020-21
4,"Sat, Dec 12",Athletico Paranaense,Atlético-MG,0 - 1,FT,Brazilian Serie A,2020-21


In [136]:
df.tail()

,Date,Home Team,Away Team,Score,Time,Competition,Season
68,"Wed, Nov 5",Atlético-MG,Bahia,3 - 0,FT,Brazilian Serie A,2025-26
69,"Wed, Oct 15",Atlético-MG,Cruzeiro,1 - 1,FT,Brazilian Serie A,2025-26
70,"Wed, Oct 8",Atlético-MG,Sport,3 - 1,FT,Brazilian Serie A,2025-26
71,"Wed, Sep 17",Bolívar,Atlético-MG,2 - 2,FT,CONMEBOL Sudamericana,2025-26
72,"Wed, Sep 24",Atlético-MG,Bolívar,1 - 0,FT,CONMEBOL Sudamericana,2025-26


In [137]:
#Rename columns for consistancy 
df.columns = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "Score",
    "Status",
    "Competition",
    "Season"
]

In [138]:
#Extarct numerical values for the away and home goals. Used for training later
df["HomeGoals"] = pd.to_numeric(df["Score"].str[0])
df["AwayGoals"] = pd.to_numeric(df["Score"].str[-1])

In [139]:
df.head()

,Date,HomeTeam,AwayTeam,Score,Status,Competition,Season,HomeGoals,AwayGoals
0,"Mon, Jan 11",Red Bull Bragantino,Atlético-MG,2 - 2,FT,Brazilian Serie A,2020-21,2,2
1,"Mon, Nov 2",Palmeiras,Atlético-MG,3 - 0,FT,Brazilian Serie A,2020-21,3,0
2,"Mon, Oct 19",Bahia,Atlético-MG,3 - 1,FT,Brazilian Serie A,2020-21,3,1
3,"Sat, Aug 22",Internacional,Atlético-MG,1 - 0,FT,Brazilian Serie A,2020-21,1,0
4,"Sat, Dec 12",Athletico Paranaense,Atlético-MG,0 - 1,FT,Brazilian Serie A,2020-21,0,1


In [140]:
df.info()

<class 'pandas.DataFrame'>
Index: 410 entries, 0 to 72
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Date         410 non-null    str  
 1   HomeTeam     410 non-null    str  
 2   AwayTeam     410 non-null    str  
 3   Score        410 non-null    str  
 4   Status       410 non-null    str  
 5   Competition  410 non-null    str  
 6   Season       410 non-null    str  
 7   HomeGoals    410 non-null    int64
 8   AwayGoals    410 non-null    int64
dtypes: int64(2), str(7)
memory usage: 58.9 KB


In [141]:
#Create a column that says whether Galo (nickname for Atletico-MG) was at home or not. 
df["IsHome"] = (df["HomeTeam"].str.contains("Atlético-MG"))

In [142]:
df.head(10)

,Date,HomeTeam,AwayTeam,Score,Status,Competition,Season,HomeGoals,AwayGoals,IsHome
0,"Mon, Jan 11",Red Bull Bragantino,Atlético-MG,2 - 2,FT,Brazilian Serie A,2020-21,2,2,False
1,"Mon, Nov 2",Palmeiras,Atlético-MG,3 - 0,FT,Brazilian Serie A,2020-21,3,0,False
2,"Mon, Oct 19",Bahia,Atlético-MG,3 - 1,FT,Brazilian Serie A,2020-21,3,1,False
3,"Sat, Aug 22",Internacional,Atlético-MG,1 - 0,FT,Brazilian Serie A,2020-21,1,0,False
4,"Sat, Dec 12",Athletico Paranaense,Atlético-MG,0 - 1,FT,Brazilian Serie A,2020-21,0,1,False
5,"Sat, Dec 26",Atlético-MG,Coritiba,2 - 0,FT,Brazilian Serie A,2020-21,2,0,True
6,"Sat, Feb 13",Atlético-MG,Bahia,1 - 1,FT,Brazilian Serie A,2020-21,1,1,True
7,"Sat, Jan 23",Vasco da Gama,Atlético-MG,3 - 2,FT,Brazilian Serie A,2020-21,3,2,False
8,"Sat, Mar 14",Villa Nova-MG,Atlético-MG,1 - 3,FT,Brazilian Campeonato Mineiro,2020-21,1,3,False
9,"Sat, Mar 7",Atlético-MG,Cruzeiro,2 - 1,FT,Brazilian Campeonato Mineiro,2020-21,2,1,True


In [143]:
df["GaloGoals"] = df["HomeGoals"].where(df["IsHome"] == 1, df["AwayGoals"])
df["OppGoals"] = df["AwayGoals"].where(df["IsHome"] == 1, df["HomeGoals"])

In [144]:
df.head()

,Date,HomeTeam,AwayTeam,Score,Status,Competition,Season,HomeGoals,AwayGoals,IsHome,GaloGoals,OppGoals
0,"Mon, Jan 11",Red Bull Bragantino,Atlético-MG,2 - 2,FT,Brazilian Serie A,2020-21,2,2,False,2,2
1,"Mon, Nov 2",Palmeiras,Atlético-MG,3 - 0,FT,Brazilian Serie A,2020-21,3,0,False,0,3
2,"Mon, Oct 19",Bahia,Atlético-MG,3 - 1,FT,Brazilian Serie A,2020-21,3,1,False,1,3
3,"Sat, Aug 22",Internacional,Atlético-MG,1 - 0,FT,Brazilian Serie A,2020-21,1,0,False,0,1
4,"Sat, Dec 12",Athletico Paranaense,Atlético-MG,0 - 1,FT,Brazilian Serie A,2020-21,0,1,False,1,0
